---
title: Install and use transpiler plugins
description: How to install and use transpiler plugins in Qiskit.
---


# Install and use transpiler plugins

{/*
  DO NOT EDIT THIS CELL!!!
  This cell's content is generated automatically by a script. Anything you add
  here will be removed next time the notebook is run. To add new content, create
  a new cell before or after this one.
*/}

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>

To facilitate the development and reuse of custom transpilation code by the wider community of Qiskit users, the Qiskit SDK supports a plugin interface that enables third-party Python packages to declare that they provide extended transpilation functionality accessible via Qiskit.

Currently, third-party plugins can provide extended transpilation functionality in three ways:

- A [transpiler stage plugin](/docs/api/qiskit/transpiler_plugins) provides a pass manager that can be used in place of one of the [6 stages](transpiler-stages) of a preset staged pass manager: `init`, `layout`, `routing`, `translation`, `optimization`, and `scheduling`.
- A [unitary synthesis plugin](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.UnitarySynthesisPlugin) provides extended functionality for unitary gate synthesis.
- A [high-level synthesis plugin](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPlugin) provides extended functionality for synthesizing "high-level objects" such as linear functions or Clifford operators. High-level objects are represented by subclasses of the [Operation](/docs/api/qiskit/qiskit.circuit.Operation) class.

The rest of the page describes how to list available plugins, install new ones, and use them.

## List available plugins and install new ones

Qiskit already includes some built-in plugins for transpilation. To install more, you can use your Python package manager. For example, you might run `pip install qiskit-toqm` to install the [Qiskit TOQM](https://github.com/qiskit-toqm/qiskit-toqm) routing stage plugin. A number of third-party plugins are part of the [Qiskit ecosystem](https://qiskit.github.io/ecosystem/#transpiler_plugin).

### List available transpiler stage plugins

Use the [list_stage_plugins](/docs/api/qiskit/transpiler_plugins#qiskit.transpiler.preset_passmanagers.plugin.list_stage_plugins) function, passing the name of the stage whose plugins you want to list.

In [1]:
from qiskit.transpiler.preset_passmanagers.plugin import list_stage_plugins

list_stage_plugins("layout")

['default', 'dense', 'sabre', 'trivial']

In [2]:
list_stage_plugins("routing")

['basic', 'default', 'lookahead', 'none', 'sabre']

 If `qiskit-toqm` were installed, then `toqm` would appear in the list of `routing` plugins.

### List available unitary synthesis plugins

Use the [unitary_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.unitary_synthesis_plugin_names) function.

In [3]:
from qiskit.transpiler.passes.synthesis import unitary_synthesis_plugin_names

unitary_synthesis_plugin_names()

['aqc', 'clifford', 'default', 'gridsynth', 'sk']

### List available high-level synthesis plugins

Use the [high_level_synthesis_plugin_names](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.high_level_synthesis_plugin_names) function, passing the name of the type of "high-level object" to be synthesized. The name corresponds to the [`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the [Operation](/docs/api/qiskit/qiskit.circuit.Operation) class representing the type of object being synthesized.

In [4]:
from qiskit.transpiler.passes.synthesis import (
    high_level_synthesis_plugin_names,
)

high_level_synthesis_plugin_names("clifford")

['ag', 'bm', 'default', 'greedy', 'layers', 'lnn']

You can use the [HighLevelSynthesisPluginManager](/docs/api/qiskit/qiskit.transpiler.passes.synthesis.plugin.HighLevelSynthesisPluginManager) class to list the names of all high-level synthesis plugins:

In [5]:
from qiskit.transpiler.passes.synthesis.plugin import (
    HighLevelSynthesisPluginManager,
)

HighLevelSynthesisPluginManager().plugins.names()

['FullAdder.default',
 'FullAdder.ripple_c04',
 'FullAdder.ripple_v95',
 'HalfAdder.default',
 'HalfAdder.qft_d00',
 'HalfAdder.ripple_c04',
 'HalfAdder.ripple_r25',
 'HalfAdder.ripple_v95',
 'IntComp.default',
 'IntComp.noaux',
 'IntComp.twos',
 'ModularAdder.default',
 'ModularAdder.modular_v17',
 'ModularAdder.qft_d00',
 'ModularAdder.ripple_c04',
 'ModularAdder.ripple_v95',
 'Multiplier.cumulative_h18',
 'Multiplier.default',
 'Multiplier.qft_r17',
 'PauliEvolution.default',
 'PauliEvolution.rustiq',
 'WeightedSum.default',
 'annotated.default',
 'clifford.ag',
 'clifford.bm',
 'clifford.default',
 'clifford.greedy',
 'clifford.layers',
 'clifford.lnn',
 'linear_function.default',
 'linear_function.kms',
 'linear_function.pmh',
 'mcmt.default',
 'mcmt.noaux',
 'mcmt.vchain',
 'mcmt.xgate',
 'mcx.1_clean_b95',
 'mcx.1_clean_kg24',
 'mcx.1_dirty_kg24',
 'mcx.2_clean_kg24',
 'mcx.2_dirty_kg24',
 'mcx.default',
 'mcx.gray_code',
 'mcx.n_clean_m15',
 'mcx.n_dirty_i15',
 'mcx.noaux_hp24'

## Use a plugin

In this section, we show how to use transpiler plugins. In the code examples, we use plugins that come with Qiskit, but plugins installed from third-party packages are used the same way.

### Use a transpiler stage plugin

To use a transpiler stage plugin, specify its name with the appropriate argument to [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). The argument is formed by appending `_method` to the name of the transpilation stage. For example, to use the `lookahead` routing plugin, we would specify `lookahead` for the `routing_method` argument.

<Admonition type="note">
The  `FakeSherbrooke` mock backend from `qiskit_ibm_runtime` is used in these examples  but you can try it on any Qiskit-compatible real or fake backend.  Your results might be different.
</Admonition>

In [6]:
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, routing_method="lookahead"
)

### Use a unitary synthesis plugin

To use a unitary synthesis plugin, specify its name as the `unitary_synthesis_method` argument to [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile):

In [7]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    backend=backend,
    unitary_synthesis_method="sk",
    unitary_synthesis_plugin_config=dict(
        basis_gates=["cz", "id", "rz", "sx", "x"]
    ),
)

Unitary synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

Use the `unitary_synthesis_plugin_config` argument, a free-form dictionary, to pass options for the unitary synthesis method. The documentation of the synthesis method should explain the options it supports. See [this list](/docs/api/qiskit/transpiler_synthesis_plugins#unitary-synthesis-plugins-2) for links to the documentation of the built-in unitary synthesis plugins.

### Use a high-level synthesis plugin

First, create an [HLSConfig](/docs/api/qiskit/qiskit.transpiler.passes.HLSConfig) to
store the names of the plugins to use for various high-level objects. For example:

In [8]:
from qiskit.transpiler.passes import HLSConfig

hls_config = HLSConfig(clifford=["layers"], linear_function=["pmh"])

This code cell creates a high-level synthesis configuration that uses the `layers` plugin
for synthesizing [Clifford](/docs/api/qiskit/qiskit.quantum_info.Clifford#clifford) objects and the `pmh` plugin for synthesizing
[LinearFunction](/docs/api/qiskit/qiskit.circuit.library.LinearFunction#linearfunction) objects. The names of the keyword arguments correspond to the
[`name`](/docs/api/qiskit/qiskit.circuit.Operation#name) attribute of the [Operation](/docs/api/qiskit/qiskit.circuit.Operation) class representing the type of object being synthesized.
For each high-level object, the list of given plugins are tried in sequence until one of them
succeeds (in the example above, each list only contains a single plugin).

In addition to specifying
a plugin by its name, you can instead pass a `(name, options)` tuple, where the second element of the tuple is a dictionary containing options for the plugin. The documentation of the synthesis method should explain the options it supports. See [this list](/docs/api/qiskit/transpiler_synthesis_plugins#high-level-synthesis-plugins-2) for links to the documentation of the built-in high-level synthesis plugins.

Once you have created the `HLSConfig` object, pass it as the
`hls_config` argument to [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile):

In [9]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, hls_config=hls_config
)

High-level synthesis is used in the `init`, `translation`, and `optimization` stages of the staged pass manager returned by [`generate_preset_pass_manager`](/docs/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) or used in [`transpile`](/docs/api/qiskit/compiler#qiskit.compiler.transpile). See [Transpiler stages](transpiler-stages) for a description of these stages.

## Next steps

<Admonition type="tip" title="Recommendation">
    - [Create a transpiler plugin](./create-transpiler-plugin).
    - Check out the [tutorials](/docs/tutorials) for examples of transpiling and running quantum circuits.
</Admonition>